# Fig. 2G, fig. S5I: cross-analysis overlap (pseudotime-dynamic ∩ perturbation-responsive)

Per lineage, the genes whose 3'UTR usage is pseudotime-dynamic (shifts along the
neurogenic trajectory, in controls) and those that are perturbation-responsive
(shift upon CRISPR knockout of the 36 ASD-risk transcription factors) overlap more
than expected by chance, i.e. the perturbation-responsive genes come from the same
APA-plastic gene set as the developmental programme. Overlap computed within the genes
testable in both analyses (co-tested universe).
Source: `dirmult_treatment_results/cross_experiment_overlap/`. Kernel: `apa-python`.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2_unweighted

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig3d_cross_experiment_overlap"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(86)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

OV = Path(
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/"
    "dirmult_treatment_results/cross_experiment_overlap"
)

# set colours: pseudotime-dynamic vs perturbation-responsive gene sets
COL_DEV = "#2c7fb8"  # pseudotime-dynamic (neurogenic trajectory, controls)
COL_TRT = "#d95f0e"  # perturbation-responsive (ASD-TF CRISPR knockout)
LINS = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]

In [ ]:
# load the co-tested gene tables (one row per gene testable in both experiments)
summ = pd.read_csv(OV / "overlap_summary.csv")
summ = summ[summ.tier == "discovery"].set_index("lineage")

sets = {}
for tag, _ in LINS:
    g = pd.read_csv(OV / f"overlap_gene_table_{tag}.csv")
    n_pseudo = int(g.sig_pseudo.sum())
    n_treat = int(g.sig_treat.sum())
    n_both = int((g.sig_pseudo & g.sig_treat).sum())
    universe = len(g)
    row = summ.loc[tag]
    sets[tag] = dict(
        n_pseudo=n_pseudo,
        n_treat=n_treat,
        n_both=n_both,
        universe=universe,
        fold=float(row.fold),
        expected=float(row.expected),
        p=float(row.fisher_p_one_sided),
        both_genes=sorted(g.loc[g.sig_pseudo & g.sig_treat, "symbol"]),
    )
    # cross-check against summary
    assert n_both == int(row.overlap), (tag, n_both, row.overlap)
    print(
        f"{tag}: pseudo={n_pseudo} treat={n_treat} both={n_both} "
        f"(exp {row.expected:.1f}, fold {row.fold:.2f}, p={row.fisher_p_one_sided:.1e}) "
        f"universe={universe}"
    )
    print("   overlap genes:", ", ".join(sets[tag]["both_genes"]))

In [ ]:
# Per-lineage area-proportional two-set Venn (pseudotime-dynamic | perturbation-responsive),
# stacked EXC over INH (circle + intersection areas scale with the counts).
from matplotlib_venn import venn2

fig, axes = plt.subplots(2, 1, figsize=FIGSIZE)


def fmt_p(p):
    if p == 0:
        return "0"
    exp = int(np.floor(np.log10(p)))
    mant = p / 10**exp
    return f"{mant:.1f}x10$^{{{exp}}}$"


for ax, (tag, label) in zip(axes, LINS):
    s = sets[tag]
    only_dev = s["n_pseudo"] - s["n_both"]
    only_trt = s["n_treat"] - s["n_both"]
    v = venn2(
        subsets=(only_dev, only_trt, s["n_both"]),
        set_labels=("Pseudotime-dynamic", "Perturbation-responsive"),
        ax=ax,
    )
    for pid, col in [("10", COL_DEV), ("01", COL_TRT), ("11", "#6a3d9a")]:
        p = v.get_patch_by_id(pid)
        if p is not None:
            p.set_color(col)
            p.set_alpha(0.55)
            p.set_edgecolor("white")
            p.set_linewidth(0.5)
    for pid in ("10", "01", "11"):
        lb = v.get_label_by_id(pid)
        if lb is not None:
            lb.set_fontsize(S.FONT_LABEL if pid == "11" else S.FONT_TICK)
            if pid == "11":
                lb.set_fontweight("bold")
    for sl, col in zip(v.set_labels, (COL_DEV, COL_TRT)):
        if sl is not None:
            sl.set_fontsize(S.FONT_ANNOT - 0.5)
            sl.set_color(col)
            sl.set_fontweight("bold")
    ax.set_title(
        label, fontsize=S.FONT_LABEL, color=S.paper_palette_lineage[tag], fontweight="bold", pad=2
    )
    ax.annotate(
        f"{s['fold']:.1f}x enriched (obs {s['n_both']} vs exp {s['expected']:.1f}); "
        f"Fisher p = {fmt_p(s['p'])}",
        xy=(0.5, -0.02),
        xycoords="axes fraction",
        ha="center",
        va="top",
        fontsize=S.FONT_ANNOT - 0.5,
    )

fig.text(
    0.5,
    0.005,
    "overlap within genes testable in both analyses "
    f"(universe {sets['excitatory']['universe']:,} EXC / {sets['inhibitory']['universe']:,} INH); "
    "areas proportional to counts",
    ha="center",
    va="bottom",
    fontsize=S.FONT_ANNOT - 1.5,
    style="italic",
    color="0.4",
)
fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.06, hspace=0.55)
S.save_and_show(fig, PANEL, figure="Fig3")

### Variants: lineage-split cross-experiment overlap Venns (half-width, eighth-height)

Single-lineage versions of the overlap Venn (excitatory and inhibitory separately) at
eighth-height; fold-enrichment + Fisher p moved into the title. Saved as
`fig3d_cross_experiment_overlap__{exc,inh}_eighth`.

In [ ]:
# --- Lineage-split cross-experiment overlap Venns (half-width, eighth-height) ---
# Single-lineage (EXC / INH) versions of the main panel. Reuses `sets` and `LINS` from above.
from matplotlib_venn import venn2 as _venn2

_COL_DEV, _COL_TRT = "#2c7fb8", "#d95f0e"


def _fmt_p3d(p):
    if p == 0:
        return "0"
    e = int(np.floor(np.log10(p)))
    m = p / 10**e
    return f"{m:.1f}x10$^{{{e}}}$"


def _venn3d(tag, save_name):
    s = sets[tag]
    linlabel = dict(LINS)[tag]
    fig, ax = plt.subplots(figsize=(S.HALF_W_IN, S.EIGHTH_H_IN))
    v = _venn2(
        subsets=(s["n_pseudo"] - s["n_both"], s["n_treat"] - s["n_both"], s["n_both"]),
        set_labels=("Pseudotime-dynamic", "Perturbation-responsive"),
        ax=ax,
    )
    for pid, col in [("10", _COL_DEV), ("01", _COL_TRT), ("11", "#6a3d9a")]:
        p = v.get_patch_by_id(pid)
        if p is not None:
            p.set_color(col)
            p.set_alpha(0.55)
            p.set_edgecolor("white")
            p.set_linewidth(0.5)
    for pid in ("10", "01", "11"):
        lb = v.get_label_by_id(pid)
        if lb is not None:
            lb.set_fontsize(S.FONT_TICK if pid == "11" else S.FONT_TICK - 1)
            if pid == "11":
                lb.set_fontweight("bold")
    for sl, col in zip(v.set_labels, (_COL_DEV, _COL_TRT)):
        if sl is not None:
            sl.set_fontsize(S.FONT_ANNOT - 1)
            sl.set_color(col)
            sl.set_fontweight("bold")
    ax.set_title(
        f"{linlabel}   ({s['fold']:.1f}x, Fisher p = {_fmt_p3d(s['p'])})",
        fontsize=S.FONT_TICK,
        color=S.paper_palette_lineage[tag],
        fontweight="bold",
        pad=2,
    )
    fig.subplots_adjust(left=0.02, right=0.98, top=0.80, bottom=0.08)
    S.save_and_show(fig, save_name, figure="Fig3")


for _tag in ("excitatory", "inhibitory"):
    _venn3d(_tag, f"fig3d_cross_experiment_overlap__{_tag[:3]}_eighth")